In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent  
src_path = project_root / "src"

sys.path.insert(0, str(project_root))

print("Project root:", project_root)
print("Src path:", src_path)


Project root: c:\Users\test\Documents\bias_lense
Src path: c:\Users\test\Documents\bias_lense\src


In [2]:
from datasets import load_dataset

dataset = load_dataset("AyoubChLin/CNN_News_Articles_2011-2022", split="train")

print(dataset)
print("Example row:")
print(dataset[0])


c:\Users\test\Documents\bias_lense\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['text', 'label'],
    num_rows: 32218
})
Example row:
{'text': ' (CNN)Celebrities such as Piers Morgan and nearly 800 other members of a London golf club will earn £85,000 ($107,100) each after agreeing the sale of their Wimbledon Park Golf Club to the organizer of the Wimbledon grand slam tennis tournament.The All England Lawn Tennis Club (AELTC) has acquired the land, which lies across the road from the tournament venue, for £65 million ($81.9 million) with a long-term view to transferring Wimbledon\'s qualifying event to the 73-acre site and building over the golf greens.The two parties have been in discussion for a decade but members of the 120-year-old golf club had resisted a number of lower bids before 82% agreed to Wimbledon\'s "best and final offer" at a meeting in central London Thursday. Very sad news.Played there for over 30yrs & voted against the sale. Hope the superb pro-shop team get properly looked after. https://t.co/Et4ATm5eMl— Piers Morgan (@

In [3]:
import re

def extract_year(row, text: str):
    for key in ["date", "publish_date", "published", "time", "timestamp", "year"]:
        if key in row and row[key]:
            m = re.search(r"(19|20)\d{2}", str(row[key]))
            if m:
                return int(m.group())

    m = re.search(r"\b(19|20)\d{2}\b", text)
    if m:
        return int(m.group())

    return None


In [7]:
TEXT_COL = "text"
CATEGORY_COL = "label"
SAMPLE_SIZE = 50

ds_small = dataset.shuffle(seed=42).select(range(SAMPLE_SIZE))

MAX_WORDS = 1200

docs = []
for row in ds_small:
    raw_text = row.get(TEXT_COL, "")
    raw_cat = row.get(CATEGORY_COL, "Unknown")

    if raw_text is None:
        continue

    text = str(raw_text).strip()
    if len(text) < 50:
        continue

    words = text.split()
    if len(words) > MAX_WORDS:
        text = " ".join(words[:MAX_WORDS])

    category = str(raw_cat)
    year = extract_year(row, text)

    docs.append(
        {
            "id": len(docs),
            "title": f"[{category}] {text[:80].replace('\n', ' ')}...",
            "text": text,
            "year": year,
        }
    )

len(docs), docs[0]


(50,
 {'id': 0,
  'title': '[3] Story highlightsA jewelry store in an upscale district of Paris was robbedAn est...',
  'text': "Story highlightsA jewelry store in an upscale district of Paris was robbedAn estimated 800,000 euros worth of luxury watches were stolenAnother jewelry store in the area was robbed in SeptemberOne of Paris' most upscale districts was once again the site of an armed jewelry store robbery Thursday, police said. Police said a number of luxury watches were stolen in a midday heist near Place Vendome, just blocks from the presidential palace and other important buildings. A couple, each armed with a handgun, entered the store and managed to escape with several luxury watches, police said. Police declined to give an estimate for the value of the stolen goods, but CNN affiliate BFMTV reported that they were worth 800,000 euros (about U.S. $1.1 million). JUST WATCHED$1.3 million reward for Cannes jewelryReplayMore Videos ...MUST WATCH$1.3 million reward for Cannes je

In [8]:
from src.embeddings import build_index

index = build_index(docs)
print("Index built with", len(index["docs"]), "documents")


Index built with 50 documents


In [9]:
from pathlib import Path
import json
import numpy as np
from sklearn.decomposition import PCA

from src.classifier import analyze_emotion
from src.summarizer import summarize
from src.embeddings import emb_model

enriched = []
for d in docs:
    text = d["text"]
    summary = summarize(text)
    emo = analyze_emotion(summary)

    enriched.append({
        "id": d["id"],
        "title": d["title"],
        "year": d.get("year"),
        "emotion_label": emo["label"],
        "happiness": float(emo["scores"]["happiness"]),
        "fear": float(emo["scores"]["fear"]),
        "motivation": float(emo["scores"]["motivation"]),
        "length": len(text.split()),
        "summary": summary,
    })

texts = [d["text"] for d in docs]
X = emb_model.encode(texts, convert_to_numpy=True, normalize_embeddings=True)
xyz = PCA(n_components=3, random_state=42).fit_transform(X)

for i, row in enumerate(enriched):
    row["x"] = float(xyz[i, 0])
    row["y"] = float(xyz[i, 1])
    row["z"] = float(xyz[i, 2])

out_dir = Path("data")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "dashboard_300.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(enriched, f, ensure_ascii=False, indent=2)

print("Saved:", out_path.resolve())


Your max_length is set to 60, but your input_length is only 37. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=18)
Your max_length is set to 60, but your input_length is only 18. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=9)


Saved: C:\Users\test\Documents\bias_lense\notebooks\data\dashboard_300.json


In [8]:
from embeddings import semantic_search
from summarizer import summarize
from classifier import analyze_emotion


def news_emotion_agent_cnn(query: str, index, k: int = 3):
    hits = semantic_search(index, query, k=k)
    enriched = []

    for h in hits:
        try:
            summary = summarize(h["text"])
            emotion = analyze_emotion(h["text"]) 
        except Exception as e:
            print("⚠️ Skipping one article due to error:", e)
            continue

        enriched.append(
            {
                "title": h["title"],
                "score": h["score"],
                "summary": summary,
                "emotion_label": emotion["label"],
                "emotion_scores": emotion["scores"],
            }
        )
    return enriched



Device set to use cpu


In [9]:
queries = [
    "elections in the United States",
    "climate change and hurricanes",
    "stock market and inflation",
]

for q in queries:
    print("\n" + "#" * 80)
    print("QUERY:", q)
    print("#" * 80)

    results = news_emotion_agent_cnn(q, index, k=3)
    for r in results:
        scores_3dp = {k: round(v, 3) for k, v in r["emotion_scores"].items()}

        print("\n" + "=" * 60)
        print("Title:", r["title"])
        print(f"Relevance score: {r['score']:.3f}")
        print("Summary:", r["summary"])
        print("Emotion:", r["emotion_label"], scores_3dp)



################################################################################
QUERY: elections in the United States
################################################################################

Title: [4] (CNN)As Florida Gov. Ron DeSantis runs for president looks to the final year of ...
Relevance score: 0.443
Summary: Florida Gov. Ron DeSantis wants to create an Office of Election Crime and Security. The office will contain 52 staff members, including 20 sworn law enforcement officers and 25 non-sworn investigators.
Emotion: happiness {'happiness': 0.71, 'fear': 0.013, 'motivation': 0.333}

Title: [4] Washington (CNN)In some corners of the political internet, there are still some ...
Relevance score: 0.417
Summary: A 38-seat loss is the third-largest change of seats in the post-Watergate era. The 8.1% spread between Democrats and Republicans is a larger percentage-point differential than in any recent wave midterm election.
Emotion: happiness {'happiness': 1.022, 'fear': 0.013

Your max_length is set to 60, but your input_length is only 18. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=9)



Title: [0] New York (CNN Business)The Labor Department's monthly jobs report can be hard to...
Relevance score: 0.343
Summary: The January jobs report came in far better than most economists had expected. The report also revised previously ho-hum figures for November and December. The Fed is likely to raise interest rates in small increments, likely a quarter of a percentage point at a time.
Emotion: happiness {'happiness': 1.117, 'fear': 0.446, 'motivation': 0.525}

Title: [0] Western countries could face oil prices of over $300 per barrel and the possible...
Relevance score: 0.249
Summary: Russian Deputy Prime Minister Alexander Novak says a rejection of Russian oil would lead to catastrophic consequences for the global market. Russia supplies 40% of Europe's gas.
Emotion: happiness {'happiness': 0.286, 'fear': 0.012, 'motivation': 0.134}

Title: [5] Story highlightsYearling sells for $3.4 millionColt sired by Dubawi and Fallen f...
Relevance score: 0.227
Summary: A total of 136 lot

In [10]:
emotional_docs = [
    {
        "id": 1,
        "title": "This policy is a complete disaster for normal people",
        "text": (
            "I am absolutely furious about the new government policy. "
            "It feels like they don't care about ordinary people at all. "
            "Everything is getting more expensive and we are the ones paying the price. "
            "This decision is unfair, selfish and honestly terrifying for our future."
        ),
    },
    {
        "id": 2,
        "title": "This breakthrough gives me real hope for the future",
        "text": (
            "This is honestly one of the most inspiring pieces of news I've read this year. "
            "It gives me real hope for the future. The people behind this project are brilliant "
            "and their work could change millions of lives for the better. "
            "I feel excited, grateful and motivated to contribute in any way I can."
        ),
    },
    {
        "id": 3,
        "title": "I’m scared about where the world is heading",
        "text": (
            "Every day I wake up and read the news and I feel more scared. "
            "Wars, crises, corruption everywhere. It feels like nobody is in control. "
            "I’m worried about my family, my friends and my own future. "
            "It's hard not to feel anxious and hopeless when things look this dark."
        ),
    },
    {
        "id": 4,
        "title": "This project motivates me to work harder than ever",
        "text": (
            "Working on this project has been the most motivating experience in my life. "
            "Even when I’m tired, I feel a huge drive to keep going, learn more and build something meaningful. "
            "I know the road will be difficult, but I’m determined to push through every obstacle and grow."
        ),
    },
    {
        "id": 5,
        "title": "I feel pure joy watching this community come together",
        "text": (
            "Seeing this community come together fills me with pure joy. "
            "People are helping each other, sharing resources, and celebrating every small success. "
            "It's heartwarming and uplifting, and it reminds me that there is still so much kindness in the world."
        ),
    },
]
len(emotional_docs)


5

In [11]:
from embeddings import build_index

emotional_index = build_index(emotional_docs)
print("Emotional index size:", len(emotional_index["docs"]))


Emotional index size: 5


In [12]:
from embeddings import semantic_search
from classifier import analyze_emotion


def news_emotion_agent_emotional(query: str, index=emotional_index, k: int = 3):
    """
    1) Semantic search on emotional_docs
    2) Classify emotion of the FULL text (no summarization)
    """
    hits = semantic_search(index, query, k=k)
    enriched = []

    for h in hits:
        emotion = analyze_emotion(h["text"])

        enriched.append(
            {
                "title": h["title"],
                "score": h["score"],
                "text": h["text"],
                "emotion_label": emotion["label"],
                "emotion_scores": emotion["scores"],
            }
        )
    return enriched


In [14]:
queries = [
    "unfair government policy",
    "hope for the future",
    "I feel scared about the world",
    "this project motivates me",
]

for q in queries:
    print("\n" + "#" * 80)
    print("QUERY:", q)
    print("#" * 80)

    results = news_emotion_agent_cnn(q, index, k=3)
    for r in results:
        scores_3dp = {k: round(v, 3) for k, v in r["emotion_scores"].items()}

        print("\n" + "=" * 60)
        print("Title:", r["title"])
        print(f"Relevance score: {r['score']:.3f}")
        print("Summary:", r["summary"])
        print("Emotion:", r["emotion_label"], scores_3dp)



################################################################################
QUERY: unfair government policy
################################################################################

Title: [3] Brisbane, Australia  (CNN)The Australian government has successfully appealed a ...
Relevance score: 0.356
Summary: The full bench of the Federal Court handed down its unanimous ruling Tuesday. The case was brought by eight Australians under 18 years old, including Melbourne teenager Anjali Sharma. The Climate Council said the ruling was disappointing but the children succeeded in drawing more attention to an important issue.
Emotion: happiness {'happiness': 0.387, 'fear': 0.042, 'motivation': 0.036}

Title: [3] Story highlightsErdogan granted sweeping new powers by referendum President tell...
Relevance score: 0.279
Summary: Erdogan says constitutional reform package is not about him. "I am a mortal really, I could die at any time," he tells CNN. Erdogan suggests a face-to-face mee

Your max_length is set to 60, but your input_length is only 40. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=20)



Title: [3] Kent Sepkowitz is a physician and infection control expert at Memorial Sloan Ket...
Relevance score: 0.350
Summary: Experts in infectious disease, epidemiology, public health, pandemic modeling and perhaps crystal ball interpretation are being trotted out to give their two cents on what lies ahead. Many turn to the story of the 1918 flu, a 15-month, three-wave experience, to inform their predictions. We should leave behind any hopes based on what happened in 1918.
Emotion: motivation {'happiness': 0.654, 'fear': 0.469, 'motivation': 0.894}

Title: [3] (CNN)Humanity needs to "grow up" and deal with the issue of climate change, Brit...
Relevance score: 0.303
Summary: British Prime Minister Boris Johnson was a last-minute addition to the speakers' list. He slammed the world's inadequate response to the climate crisis. "My friends, the adolescence of humanity is coming to an end," he said.
Emotion: happiness {'happiness': 0.419, 'fear': 0.013, 'motivation': 0.321}

Title: [3] K

Your max_length is set to 60, but your input_length is only 40. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=20)



Title: [3] (CNN)Humanity needs to "grow up" and deal with the issue of climate change, Brit...
Relevance score: 0.410
Summary: British Prime Minister Boris Johnson was a last-minute addition to the speakers' list. He slammed the world's inadequate response to the climate crisis. "My friends, the adolescence of humanity is coming to an end," he said.
Emotion: happiness {'happiness': 0.419, 'fear': 0.013, 'motivation': 0.321}

Title: [3] Story highlightsScientists and journalists are worried about the condition of th...
Relevance score: 0.330
Summary: The burning of coal, oil, and natural gas has made the planet warmer. World on a trajectory that would lead to an increase of 4C (7F) in this century. Methane release from thawing permafrost in the Arctic "is the most dangerous amplifying feedback in the entire carbon cycle"
Emotion: happiness {'happiness': 0.71, 'fear': 0.619, 'motivation': 0.678}

Title: [3] Kent Sepkowitz is a physician and infection control expert at Memorial Sloan Ket

Your max_length is set to 60, but your input_length is only 39. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=19)



Title: [3] CNN iReport celebrates the British monarchy ahead of the Diamond Jubilee this we...
Relevance score: 0.193
Summary: CNN iReport celebrates the British monarchy ahead of the Diamond Jubilee this weekend from June 2-5. Have you bumped into a royal? Popped into Buckingham Palace for tea? We want to hear from you about your experiences meeting the royal family.
Emotion: happiness {'happiness': 0.989, 'fear': 0.006, 'motivation': 0.836}

Title: [3] Story highlightsScientists and journalists are worried about the condition of th...
Relevance score: 0.192
Summary: The burning of coal, oil, and natural gas has made the planet warmer. World on a trajectory that would lead to an increase of 4C (7F) in this century. Methane release from thawing permafrost in the Arctic "is the most dangerous amplifying feedback in the entire carbon cycle"
Emotion: happiness {'happiness': 0.71, 'fear': 0.619, 'motivation': 0.678}

Title: [5] (CNN)Puky Ramokoatsi believes football helped saved her life.